# Notebook 06: Synthetic Arm Evaluation

**Purpose**: Run the full evaluation matrix on synthetic tasks. Answers RQ2 and RQ4.

## Why this notebook exists (thesis framing)

This is the synthetic-arm mirror of Notebooks 02+03 combined — same frozen-LLM ICL mechanism, same demonstration-selection question, but on tasks where ground truth (π_true, the true causal features) is known by construction (Notebook 04), which real TableShift data can never provide. That's what makes this notebook able to answer **RQ4** (does SATA improve *correctness* of reliance, not just accuracy?) in a way Notebook 03 structurally cannot — Notebook 03 can only measure *internal consistency* (does the model's stated reasoning match its behaviour?), not whether that behaviour is actually right.

It also re-runs **RQ2**'s question (which demonstration diversity matters for which shift type?) on tasks where the shift type is exactly known, rather than inferred from a benchmark's metadata — a clean re-test of the real-arm finding from Notebook 02 under conditions with no ambiguity about what changed between train and test.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, set_seed, resolve_path

config = load_config()
set_seed(config.seed_accuracy[0])

## Conditions (10 total)

1. Zero-shot
2. Random-k
3. Similarity-k
4. Label diversity
5. Feature-range diversity
6. Rule diversity (ground-truth regime labels — no decision tree needed)
7. Counter-spurious diversity (ground-truth is_counter_spurious tags)
8. Best protocol + SATA selection
9. SATA alone (full pool, no protocol pre-filtering)
10. SATA query-agnostic ablation

Conditions 1–7 mirror Notebook 02 exactly (same lit-review motivation — see that notebook's Conditions cell) except rule diversity and counter-spurious diversity now use the generator's *ground-truth* regime/is_counter_spurious tags directly, rather than approximating them with a fitted decision tree or a correlation search the way the real arm has to. Conditions 8–10 are the SATA-specific comparisons that operationalise RQ4:

- **"Best protocol + SATA"** doesn't mean SATA re-ranks the whole 256-row pool — it pre-filters with whichever protocol won Notebook 05's Gate 2 (determined empirically at Gate 2 time, not assumed here), then lets SATA pick the final k from within that larger candidate set. This tests whether SATA adds value *on top of* the best hand-designed heuristic, per `src/selection/sata_select.py`'s `pre_filtered_idx` design.
- **"SATA alone"** scores the entire pool with no protocol pre-filter, testing whether SATA's learned reweighting is sufficient by itself.
- **"SATA query-agnostic"** is the ablation from Notebook 05, included here specifically so its *downstream LLM accuracy* — not just its XGBoost proxy score — can be compared against full SATA. A gap between full SATA and this ablation at the LLM level is stronger evidence for query-conditioning mattering than the proxy metric alone.

In [2]:
import json

import numpy as np
import pandas as pd
import torch

from src.models.sata import SATA, SATAQueryAgnostic
from src.selection import sata_select

sata_model = SATA(
    n_features=config.generator.n_features, d_model=config.sata.d_model,
    n_heads=config.sata.n_heads, n_layers=config.sata.n_layers,
)
sata_model.load_state_dict(torch.load(resolve_path('models/sata_best.pt'), weights_only=True))
sata_model.eval()

sata_qa_model = SATAQueryAgnostic(
    n_features=config.generator.n_features, d_model=config.sata.d_model,
    n_heads=config.sata.n_heads, n_layers=config.sata.n_layers,
)
sata_qa_model.load_state_dict(torch.load(resolve_path('models/sata_query_agnostic.pt'), weights_only=True))
sata_qa_model.eval()

print("Loaded sata_best.pt and sata_query_agnostic.pt")

Loaded sata_best.pt and sata_query_agnostic.pt


## RQ2 evaluation: protocol x shift type grid

Accuracy on the 200 held-out test tasks, for each condition x shift type x k (`k_primary` and `k_sensitivity`, mirroring Notebook 02's real-arm sweep). **RQ2 success** = interaction effect: different protocols win on different shift types. Shift type is known by construction — no DISDE decomposition needed.

**Why "no DISDE decomposition needed" is worth spelling out.** On real TableShift data (Notebook 02), a protocol's accuracy gain can't automatically be attributed to a specific shift type — the shift attribution procedure the spec calls for (Liu et al. 2023, Lit-review §3, DISDE: decomposing the accuracy gap into covariate and conditional components) exists precisely because real shifts are naturally-occurring and mixed. Here, each column of this grid *is* a controlled instantiation of a single shift concept by construction (Notebook 04) — `covariate` only moves `P(x)`, `spurious_reversal`/`mechanism` only move `P(y|x)` — so the grid itself already gives the attribution the DISDE procedure would otherwise have to estimate. This is what makes the synthetic arm the clean test of RQ2's "different protocols win on different shift types" claim: any asymmetry in this grid is a real interaction effect, not an artefact of overlapping shift types muddying the picture.

**k-sensitivity**: `results/rq2_grid.parquet` stays k_primary-only (the headline number every other cell/notebook consumes); the full k_primary-vs-k_sensitivity comparison across all 10 conditions is saved separately to `results/rq2_grid_k_sensitivity.parquet` and printed below.

In [ ]:
from tqdm import tqdm

from src.data.serialisation import serialise_row
from src.data.tableshift_loader import select_top_features
from src.inference.llm_runner import VLLMWorkerRunner
from src.inference.prompts import build_classification_prompt
from src.selection import random_select, similarity_select, label_diversity, feature_range, rule_diversity, counter_spurious
from src.utils.results_schema import append_results, load_results
from src.evaluation.accuracy import summarise

SHIFT_TYPES = ['id', 'covariate', 'spurious_reversal', 'extrapolation', 'missing_feature', 'mechanism']
FEATURE_COLS = [f'feature_{i}' for i in range(config.generator.n_features)]
LABEL_TOKENS = ('0', '1')
TASK_DESCRIPTION = "the label of a synthetic binary classification task"

SYN_ROOT = resolve_path(config.paths.data_synthetic)
RESULTS_PATH = resolve_path('results/synthetic_evaluation.parquet')

CONDITIONS = [
    'zero_shot', 'random', 'similarity', 'label_diversity', 'feature_range',
    'rule_diversity', 'counter_spurious', 'best_protocol_sata', 'sata_alone', 'sata_query_agnostic',
]

# k-sensitivity sweep for the RQ2 grid (mirrors Notebook 02): all 10
# conditions, including the SATA variants, at both k_primary (headline) and
# k_sensitivity. K_VALUES[0] must stay k_primary -- zero-shot dedup and the
# "headline" rq2_grid filter both key off it.
K_VALUES = (config.k_primary, config.k_sensitivity)

try:
    import vllm  # noqa: F401
    VLLM_AVAILABLE = True
except ImportError:
    VLLM_AVAILABLE = False
    print("vLLM not installed in this environment — skipping synthetic-arm inference. "
          "Run this notebook on a GPU box with vllm + the model weights available.")

# "Best protocol" per Notebook 05's Gate 2 proxy comparison -- used as the
# pre-filter for the "best protocol + SATA" condition (see
# src/selection/sata_select.py's pre_filtered_idx docstring).
try:
    gate2 = pd.read_parquet(resolve_path('results/sata_gate2_summary.parquet'))
    BEST_PROTOCOL = gate2[gate2.method != 'sata'].sort_values('proxy_accuracy', ascending=False).iloc[0]['method']
except FileNotFoundError:
    BEST_PROTOCOL = 'counter_spurious'
print(f"Best protocol (from Notebook 05 Gate 2): {BEST_PROTOCOL}")


def select_ground_truth_protocol(protocol, pool, query, k, seed, top3_continuous):
    if protocol == 'random':
        return random_select.select(pool, query, k, seed)
    if protocol == 'label_diversity':
        return label_diversity.select(pool, query, k, seed)
    if protocol == 'feature_range':
        return feature_range.select(pool, query, k, seed, top_features=top3_continuous)
    if protocol == 'rule_diversity':
        return rule_diversity.select(pool, query, k, seed, regimes=pool['regime'])
    if protocol == 'counter_spurious':
        return counter_spurious.select(pool, query, k, seed, is_counter_spurious=pool['is_counter_spurious'])
    raise ValueError(f"Unknown ground-truth protocol: {protocol}")


def precompute_similarity_demo_ids(pool, queries, k):
    """Similarity is deterministic (no seed dependency) -- one batched
    encode() call for every query in `queries` against `pool`, instead of
    select_demos_synthetic's per-query path embedding one query at a time.
    Returns a dict keyed by `queries`' index (matching each row's `.name`).

    Pass k=max(K_VALUES) and slice the result per k in the caller: a cosine-
    similarity argsort doesn't change when you later take a smaller prefix
    of it, so this only needs computing once per (pool, queries) regardless
    of how many k values are being swept.
    """
    pool_texts = [
        serialise_row({f: pool.loc[i, f] for f in FEATURE_COLS}, label=str(int(pool.loc[i, 'label'])))
        for i in pool.index
    ]
    query_texts = [serialise_row({f: row[f] for f in FEATURE_COLS}) for _, row in queries.iterrows()]
    local_idx_per_query = similarity_select.select_batch(pool_texts, query_texts, k)
    return {
        query_id: [pool.index[i] for i in local_idx]
        for query_id, local_idx in zip(queries.index, local_idx_per_query)
    }


def select_demos_synthetic(condition, pool, query, k, seed, top3_continuous, similarity_demo_ids=None):
    if condition == 'zero_shot':
        return []
    if condition == 'similarity':
        return similarity_demo_ids[query.name][:k]
    if condition in ('random', 'label_diversity', 'feature_range', 'rule_diversity', 'counter_spurious'):
        return select_ground_truth_protocol(condition, pool, query, k, seed, top3_continuous)
    if condition == 'best_protocol_sata':
        prefilter_k = min(len(pool), 4 * k)
        candidates = select_ground_truth_protocol(BEST_PROTOCOL, pool, query, prefilter_k, seed, top3_continuous)
        return sata_select.select(sata_model, pool, query, FEATURE_COLS, 'label', k, pre_filtered_idx=candidates)
    if condition == 'sata_alone':
        return sata_select.select(sata_model, pool, query, FEATURE_COLS, 'label', k)
    if condition == 'sata_query_agnostic':
        return sata_select.select(sata_qa_model, pool, query, FEATURE_COLS, 'label', k)
    raise ValueError(f"Unknown condition: {condition}")


def build_demo_lines(pool, demo_ids):
    return [
        serialise_row({f: pool.loc[i, f] for f in FEATURE_COLS}, label=str(int(pool.loc[i, 'label'])))
        for i in demo_ids
    ]


def build_query_line(query):
    return serialise_row({f: query[f] for f in FEATURE_COLS})


def evaluate_task_group(task_paths, model_cfg, runner, conditions=CONDITIONS, k_values=(config.k_primary,)):
    task_bar = tqdm(task_paths, desc=f"Tasks ({model_cfg.name})", leave=False)
    for task_path in task_bar:
        task_df = pd.read_parquet(task_path)
        task_id = task_path.stem
        task_bar.set_postfix(task=task_id)

        # Batch every (shift_type, k, condition) combo for this task into a
        # single runner.batch_predict call instead of one call per combo --
        # with queries_per_env=32 and ~19 k/condition combos x 6 shift types,
        # per-combo calls paid fixed subprocess-IPC/kernel-launch overhead
        # ~114 times per task (~22,800 times across all 200 test tasks).
        # Batching per-task cuts that to one call per task (~3,600 rows,
        # comparable to Notebook 02's batch sizes) with identical results --
        # this only changes how many calls carry the same prompts/rows.
        task_prompts, task_batch_rows = [], []
        for shift_type in SHIFT_TYPES:
            env_df = task_df[task_df['environment'] == shift_type]
            pool = env_df[env_df['split'] == 'demo'].reset_index(drop=True)
            queries = env_df[env_df['split'] == 'query'].reset_index(drop=True)
            top3_continuous = select_top_features(pool[FEATURE_COLS + ['label']], n_features=3)

            similarity_demo_ids = (
                precompute_similarity_demo_ids(pool, queries, max(k_values))
                if 'similarity' in conditions else None
            )
            for k in k_values:
                sliced_similarity_ids = (
                    {qid: ids[:k] for qid, ids in similarity_demo_ids.items()}
                    if similarity_demo_ids is not None else None
                )
                for condition in conditions:
                    # Zero-shot has no demos, so it's identical at every k --
                    # only run it once, at the primary k, rather than
                    # duplicating identical rows per k value.
                    if condition == 'zero_shot' and k != k_values[0]:
                        continue
                    seed = config.seed_accuracy[0]
                    for query_id, query in queries.iterrows():
                        demo_ids = select_demos_synthetic(condition, pool, query, k, seed, top3_continuous, sliced_similarity_ids)
                        prompt = build_classification_prompt(
                            TASK_DESCRIPTION, LABEL_TOKENS, build_demo_lines(pool, demo_ids), build_query_line(query)
                        )
                        task_prompts.append(prompt)
                        task_batch_rows.append({
                            'arm': 'synthetic', 'dataset': task_id, 'environment': shift_type,
                            'model': model_cfg.name, 'method': condition, 'seed': int(seed),
                            'query_id': int(query_id), 'label': str(int(query['label'])),
                            'demo_ids': [int(i) for i in demo_ids], 'k': len(demo_ids),
                        })

        predictions = runner.batch_predict(task_prompts, LABEL_TOKENS)
        for row, pred in zip(task_batch_rows, predictions):
            row['prediction'] = pred.prediction
            row['logprob_0'] = pred.logprob_0
            row['logprob_1'] = pred.logprob_1
        append_results(pd.DataFrame(task_batch_rows), RESULTS_PATH)
        tqdm.write(f"{model_cfg.name} | {task_id}: done")


test_task_paths = sorted((SYN_ROOT / 'tasks_test').glob('*.parquet'))

for model_cfg in (config.base_llms if VLLM_AVAILABLE else []):
    runner = VLLMWorkerRunner(model_cfg.path, **vars(config.vllm))
    evaluate_task_group(test_task_paths, model_cfg, runner, k_values=K_VALUES)
    runner.shutdown()

# RQ2 grid: accuracy per (method, shift_type, k), averaged over test tasks.
# 'k' is included in group_cols -- without it, the k_primary and
# k_sensitivity sweep rows would get silently averaged together.
RQ2_COLS = ['method', 'shift_type', 'model', 'k', 'accuracy', 'macro_f1', 'invalid_rate', 'n']
try:
    synthetic_results = load_results(RESULTS_PATH)
except FileNotFoundError:
    synthetic_results = None

if synthetic_results is None or synthetic_results.empty:
    print("No synthetic-arm results yet — skipped (vLLM not available in this environment).")
    rq2_grid_all_k = pd.DataFrame(columns=RQ2_COLS)
else:
    rq2_source = synthetic_results[synthetic_results['dataset'].str.startswith('test_')]
    rq2_grid_all_k = summarise(rq2_source, group_cols=['method', 'environment', 'model', 'k'])
    rq2_grid_all_k = rq2_grid_all_k.rename(columns={'environment': 'shift_type'})

# Headline rq2_grid.parquet stays k_primary-only -- exactly the shape every
# downstream consumer (Notebook 08's figures/tables) already expects. The
# full k-sensitivity sweep is saved separately rather than folded in here.
rq2_grid = rq2_grid_all_k[rq2_grid_all_k['k'] == config.k_primary].drop(columns='k') if not rq2_grid_all_k.empty else rq2_grid_all_k.drop(columns='k')
rq2_grid.to_parquet(resolve_path('results/rq2_grid.parquet'), index=False)
rq2_grid_all_k.to_parquet(resolve_path('results/rq2_grid_k_sensitivity.parquet'), index=False)

if not rq2_grid_all_k.empty and len(rq2_grid_all_k['k'].unique()) > 1:
    print("k-sensitivity (accuracy, k_primary vs k_sensitivity):")
    display(rq2_grid_all_k.pivot_table(index=['model', 'method', 'shift_type'], columns='k', values='accuracy'))

if rq2_grid.empty:
    rq2_grid
else:
    rq2_grid.pivot_table(index=['model', 'method'], columns='shift_type', values='accuracy')

## RQ4 evaluation: SATA vs protocols

Compare SATA + best protocol vs. best protocol alone on:
- Accuracy (all shift types, held-out test + held-out family tasks)
- Correctness-of-reliance faithfulness: rho(pi_behav, pi_true)

**Why both accuracy *and* faithfulness, on the *same* comparison?** This is the crux of RQ4 (Lit-review §3): "SATA combined with the best-performing protocol outperforms that protocol alone on **both** R-AUC and ρ(π_self, π_behav)." Improving accuracy alone would only replicate what Notebook 05's Gate 2 already checks with the cheap XGBoost proxy. What Gate 2 *can't* check — because it has no access to an LLM's stated feature ranking or causal ground truth — is whether SATA's demonstrations lead the frozen LLM toward *correct* reliance, not just correct predictions. A configuration that improves accuracy without improving ρ(π_behav, π_true) would be exactly the RQ3-style dissociation (accuracy up, faithfulness flat) the lit review flags as a real risk (§2.5.2) — predictive gains that don't reflect genuinely better task understanding.

**Why held-out test tasks *and* held-out-family tasks, not just one?** `tasks_test` shares SATA's training rule families (linear/threshold/tree), so strong performance there could just mean SATA memorised patterns specific to those families. `tasks_heldout_family` (sparse_interaction, entirely unseen during meta-training — see Notebook 04) is the harder generalisation test: if SATA's advantage holds there too, it's evidence the selector learned something about demonstration relevance *in general*, not something tied to the specific rule shapes it was trained on.

In [ ]:
from src.evaluation.faithfulness import hot_deck_impute_feature, compute_accuracy_drop
from src.evaluation.faithfulness_correctness import true_importance_scores, correctness_rho_from_scores

RQ4_CONDITIONS = ['best_protocol_sata', BEST_PROTOCOL]
heldout_family_paths = sorted((SYN_ROOT / 'tasks_heldout_family').glob('*.parquet'))

# Accuracy: test tasks (already covered by the RQ2 pass above) + held-out
# *family* tasks (sparse_interaction, never seen during SATA training) --
# only need to additionally run inference on the latter, for just these 2
# conditions.
for model_cfg in (config.base_llms if VLLM_AVAILABLE else []):
    runner = VLLMWorkerRunner(model_cfg.path, **vars(config.vllm))
    evaluate_task_group(heldout_family_paths, model_cfg, runner, conditions=RQ4_CONDITIONS)
    runner.shutdown()

RQ4_ACC_COLS = ['task_group', 'method', 'shift_type', 'model', 'accuracy', 'macro_f1', 'invalid_rate', 'n']
try:
    synthetic_results = load_results(RESULTS_PATH)
except FileNotFoundError:
    synthetic_results = None

if synthetic_results is None or synthetic_results.empty:
    rq4_accuracy = pd.DataFrame(columns=RQ4_ACC_COLS)
else:
    # RQ4_CONDITIONS' methods were also run at k_sensitivity in the RQ2 pass
    # above (it sweeps all 10 conditions); RQ4 itself only ever evaluates at
    # k_primary, so without this filter the two 'test' task_group rows would
    # silently blend k_primary and k_sensitivity accuracy together.
    rq4_accuracy_source = synthetic_results[
        synthetic_results['method'].isin(RQ4_CONDITIONS) & (synthetic_results['k'] == config.k_primary)
    ].copy()
    rq4_accuracy_source['task_group'] = np.where(
        rq4_accuracy_source['dataset'].str.startswith('heldout_'), 'heldout_family', 'test'
    )
    rq4_accuracy = summarise(rq4_accuracy_source, group_cols=['task_group', 'method', 'environment', 'model'])
    rq4_accuracy = rq4_accuracy.rename(columns={'environment': 'shift_type'})

# Correctness-of-reliance faithfulness: rho(pi_behav, pi_true) per task, via
# the same LOO hot-deck ablation as Notebook 03, computed on the 'id'
# environment's query set for each condition. Sampled (not all 250 tasks --
# each task needs 1 + n_features reruns per condition) for tractability.
FAITHFULNESS_TASK_SAMPLE = 50
rng = np.random.default_rng(config.seed_faithfulness[0])
all_task_paths = test_task_paths + heldout_family_paths
faith_task_paths = list(rng.choice(
    all_task_paths, size=min(FAITHFULNESS_TASK_SAMPLE, len(all_task_paths)), replace=False
)) if VLLM_AVAILABLE else []

CORRECTNESS_COLS = ['task_id', 'model', 'method', 'rho', 'pval']
correctness_rows = []
for model_cfg in (config.base_llms if VLLM_AVAILABLE else []):
    runner = VLLMWorkerRunner(model_cfg.path, **vars(config.vllm))

    task_bar = tqdm(faith_task_paths, desc=f"RQ4 faithfulness tasks ({model_cfg.name})", leave=False)
    for task_path in task_bar:
        task_df = pd.read_parquet(task_path)
        task_id = task_path.stem
        task_bar.set_postfix(task=task_id)
        meta = json.load(open(task_path.parent / f'{task_id}_meta.json'))

        id_df = task_df[task_df['environment'] == 'id']
        pool = id_df[id_df['split'] == 'demo'].reset_index(drop=True)
        queries = id_df[id_df['split'] == 'query'].reset_index(drop=True)
        top3_continuous = select_top_features(pool[FEATURE_COLS + ['label']], n_features=3)
        similarity_demo_ids = (
            precompute_similarity_demo_ids(pool, queries, config.k_primary)
            if 'similarity' in RQ4_CONDITIONS else None
        )

        # Family-aware true-importance scores (see faithfulness_correctness.py):
        # only 'linear' actually reads `coefficients`, so `thresholds3`/
        # `leaf_labels` (via .get -- None for families that don't set them)
        # are needed to score 'threshold'/'tree' correctly rather than by an
        # unused random coefficient vector.
        true_scores = true_importance_scores(
            meta['rule_family'], config.generator.n_features, meta['causal_features'],
            np.array(meta['coefficients']), thresholds3=meta.get('thresholds3'), leaf_labels=meta.get('leaf_labels'),
        )

        for condition in RQ4_CONDITIONS:
            seed = config.seed_faithfulness[0]
            demo_ids_per_query = [
                select_demos_synthetic(condition, pool, row, config.k_primary, seed, top3_continuous, similarity_demo_ids)
                for _, row in queries.iterrows()
            ]

            def run_inference(df):
                prompts = [
                    build_classification_prompt(
                        TASK_DESCRIPTION, LABEL_TOKENS, build_demo_lines(pool, demo_ids), build_query_line(row)
                    )
                    for (_, row), demo_ids in zip(df.iterrows(), demo_ids_per_query)
                ]
                preds = runner.batch_predict(prompts, LABEL_TOKENS)
                return np.array([p.prediction == str(int(row['label'])) for p, (_, row) in zip(preds, df.iterrows())])

            original_correct = run_inference(queries)
            deltas = {}
            for feature in FEATURE_COLS:
                modified = hot_deck_impute_feature(queries, feature, pool, FEATURE_COLS, seed=seed)
                modified_correct = run_inference(modified)
                deltas[feature] = compute_accuracy_drop(original_correct, modified_correct)

            # Raw per-feature delta scores, not a hand-built rank permutation
            # -- spearmanr rank-transforms internally (average ranks for
            # ties), which is the right way to handle the many legitimate
            # zero-importance ties in `true_scores`.
            behav_scores = np.array([deltas[f'feature_{j}'] for j in range(config.generator.n_features)])
            result = correctness_rho_from_scores(true_scores, behav_scores)
            correctness_rows.append({
                'task_id': task_id, 'model': model_cfg.name, 'method': condition,
                'rho': result['rho'], 'pval': result['pval'],
            })

    runner.shutdown()

correctness_df = pd.DataFrame(correctness_rows, columns=CORRECTNESS_COLS)

if not VLLM_AVAILABLE:
    print("Skipped RQ4/faithfulness inference — vLLM not installed in this environment.")

if correctness_df.empty:
    rq4_faithfulness = pd.DataFrame(columns=['model', 'method', 'rho_mean', 'rho_std'])
else:
    rq4_faithfulness = correctness_df.groupby(['model', 'method'])['rho'].agg(['mean', 'std']).reset_index()
    rq4_faithfulness = rq4_faithfulness.rename(columns={'mean': 'rho_mean', 'std': 'rho_std'})

if rq4_accuracy.empty:
    rq4_comparison = rq4_accuracy.assign(rho_mean=[], rho_std=[])
else:
    rq4_comparison = rq4_accuracy.merge(rq4_faithfulness, on=['model', 'method'], how='left')

rq4_comparison.to_parquet(resolve_path('results/rq4_comparison.parquet'), index=False)
correctness_df.to_parquet(resolve_path('results/faithfulness_synthetic.parquet'), index=False)

rq4_comparison

## RQ3 correctness-of-reliance (synthetic only)

This is the strongest version of the faithfulness question the project asks, and it only exists on the synthetic arm. Notebook 03's ρ(π_self, π_behav) checks *internal consistency* — does the model's self-report match its own behaviour? — but says nothing about whether that behaviour is actually *correct*, because no real dataset can hand us π_true. Here, `true_importance_scores` derives π_true directly from the generator's own causal structure — family-aware, since `SyntheticTask._apply_rule` only reads `coefficients` for the `linear` family: `threshold` scores its thresholded features, `tree` scores each causal feature by its Boolean influence on the leaf function, `sparse_interaction` scores its two interacting features, and the spurious/noise features plus any unused causal-family "decoy" always score 0 — the one thing that's impossible to obtain on TableShift data. ρ(π_behav, π_true) therefore asks the sharper question: not just "is the model consistent with itself," but "does the model rely on the features that actually determine the label."

In [ ]:
# rho(pi_behav, pi_true) is computed once, per condition/task, in the RQ4 cell
# above (its faithfulness component *is* this RQ3 computation -- the spec
# describes them identically) and saved to faithfulness_synthetic.parquet.
# This cell just presents the summary view.
if correctness_df.empty:
    print("No correctness-of-reliance results yet (see RQ4 cell above).")
else:
    correctness_df.groupby(['model', 'method'])['rho'].describe()

## Output

- `results/synthetic_evaluation.parquet`
- `results/rq2_grid.parquet`
- `results/rq4_comparison.parquet`
- `results/faithfulness_synthetic.parquet`